# Late Interaction using ColBERT
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring server-side.<br>
However, turbopuffer doesn't support multi-vector documents or server-side MaxSim ranking today.

Since turbopuffer does not natively support it, a practical ColBERT implementation today is a two-stage approximation:
* server-side dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation decisions:
* At index time, ColBERT is run on Every document and token vectors are stored as a JSON attribute in each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer returns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.

Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors travel with the document, no lookups or joins.
* Storing JSON blobs is relatively cheaper in turbopuffer's storage model.


### Step 1: Setup
Imports, load .env, initialize turbopuffer client

In [27]:
import os
import json
import numpy as np
import turbopuffer
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-west1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-late-interaction-msmacro')

### Step 2: Dataset
Load [Quora Question Paris](https://huggingface.co/datasets/sentence-transformers/quora-duplicates) from HuggingFace

In [6]:
from datasets import load_dataset

dataset = load_dataset("microsoft/ms_marco", "v2.1", split="train", streaming=True)
df = pd.DataFrame(list(dataset.take(10000)))
print(df.count())
print(df.columns)
print(df.head(2))

answers              10000
passages             10000
query                10000
query_id             10000
query_type           10000
wellFormedAnswers    10000
dtype: int64
Index(['answers', 'passages', 'query', 'query_id', 'query_type',
       'wellFormedAnswers'],
      dtype='str')
                                             answers  \
0  [The immediate impact of the success of the ma...   
1  [Restorative justice that fosters dialogue bet...   

                                            passages  \
0  {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]...   
1  {'is_selected': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0]...   

                                               query  query_id   query_type  \
0  )what was the immediate impact of the success ...   1185869  DESCRIPTION   
1  _________ justice is designed to repair the ha...   1185868  DESCRIPTION   

  wellFormedAnswers  
0                []  
1                []  


In [4]:
# look at one row's passages
import json
print(df['passages'][0])
print(type(df['passages'][0]))

{'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade.', 'The Manhattan Project was the name for a project conducted during World War II, to develop the first atomic bomb. It refers specifically to the period of the project from 1

### Dataset: MS MARCO v2.1

[MS MARCO](https://huggingface.co/datasets/microsoft/ms_marco) (Microsoft Machine Reading Comprehension) is a large-scale dataset of real Bing search queries paired with relevant passages from web documents. It is the standard benchmark dataset that ColBERT was originally evaluated on in the 2020 paper.

**Why MS MARCO**
- Queries are real search engine queries 
- Documents are multi-sentence passages 
- Contains multiple competing ideas per passage - ideal for ColBERT's token-level matching
- Ground truth relevance judgments are provided via `is_selected` flag

**Dataset structure (per row):**
- `query` — the search query (e.g. "what was the immediate impact of the Manhattan Project")
- `query_id` — unique query identifier
- `passages` — dict containing:
  - `passage_text` — list of 10 candidate passages
  - `is_selected` — list of 0/1 flags, 1 = correct answer passage
- `answers` — human written answer (not used in this experiment)
- `query_type` — DESCRIPTION, NUMERIC, ENTITY, PERSON, LOCATION

**How we use it:**
- **Corpus** — all unique passages across 10,000 rows (~100k passages, we use a subset)
- **Queries** — the `query` field from each row
- **Ground truth** — the passage where `is_selected=1` for each query
- **Evaluation** — did the correct passage appear in top K results?

In [ ]:
corpus = []
ground_truth = {}  # query_id -> correct passage text

for _, row in df.iterrows():
    passages = row['passages']
    query_id = row['query_id']
    
    for i, passage in enumerate(passages['passage_text']):
        corpus.append(passage)
        if passages['is_selected'][i] == 1:
            ground_truth[query_id] = passage

# deduplicate corpus
corpus = list(set(corpus))

print(f"Total unique passages: {len(corpus)}")
print(f"Queries with ground truth: {len(ground_truth)}")
print(f"\nSample passage: {corpus[0][:200]}")
print(f"\nSample passage length: {len(corpus[0])}")

Total unique passages: 98835
Queries with ground truth: 6050

Sample passage: What made you want to look up och? Please tell us where you read or heard it (including the quote, if possible).

Sample passage length: 282


### Corpus Sampling Strategy

MS MARCO contains ~98K unique passages across 10K rows. Embedding all 98K passages 
on CPU would be feasible but slow. We sample down to 10K passages while guaranteeing 
that the correct answer passage for every query is included in the corpus.

Without this guarantee, evaluation would be meaningless.<br>
**Sampling approach:**
1. Extract all ground truth passages (correct answers marked with `is_selected=1`)
2. Fill remaining slots with randomly sampled passages from the full corpus
3. Shuffle to avoid ground truth passages clustering at the front of the index

This gives us a realistic retrieval challenge, 10K passages, known correct answers, 
and enough noise passages to stress-test both dense and ColBERT retrieval.

In [17]:
import random

# ensure all ground truth passages are in the corpus
ground_truth_passages = list(set(ground_truth.values()))

# fill remaining slots with random passages
remaining = [p for p in corpus if p not in ground_truth_passages]
sample_size = 10000 - len(ground_truth_passages)
sampled_corpus = ground_truth_passages + random.sample(remaining, sample_size)

# shuffle so ground truth passages aren't all at the front
random.shuffle(sampled_corpus)

print(f"Ground truth passages: {len(ground_truth_passages)}")
print(f"Total corpus size: {len(sampled_corpus)}")

# build a text to ID mapping for evaluation
text_to_id = {text: i + 1 for i, text in enumerate(sampled_corpus)}

ground_truth_ids = {query_id: text_to_id[text] 
                    for query_id, text in ground_truth.items() 
                    if text in text_to_id}

print(f"Queries with ground truth in corpus: {len(ground_truth_ids)}")

Ground truth passages: 6049
Total corpus size: 10000
Queries with ground truth in corpus: 6050


### Step 3: Embedding
Load ColBERT model via fastembed, generate dense vector and token vectors per document.<br>
> We choose [`fastembed`](https://huggingface.co/ferrisS/harrier-oss-v1-270m-fastembed) because it is a lightweight embedding library with native late interaction support, and runs on CPU.

In [19]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding

dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
colbert_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

In [24]:
# Loop over all 10K corpus passages, generate dense vectors and token vectors.

import base64

records = []

for i in range(len(sampled_corpus)):
    text = sampled_corpus[i]
    
    # dense vector from MiniLM - for ANN retrieval
    dense_vector = list(dense_model.embed([text]))[0].tolist()
    
    # token vectors from ColBERT - base64 encoded float32
    colbert_embedding = list(colbert_model.embed([text]))[0]
    token_vectors_bytes = colbert_embedding.astype(np.float32).tobytes()
    token_vectors_b64 = base64.b64encode(token_vectors_bytes).decode('utf-8')
    
    record = {
        "id": i + 1,
        "vector": dense_vector,
        "token_vectors": token_vectors_b64,
        "text": text
    }
    
    records.append(record)

print(f"Embedded {len(records)} documents")
print(f"Dense vector dims: {len(records[0]['vector'])}")

Embedded 10000 documents
Dense vector dims: 384


### Step 4: Index
Upsert to turbopuffer:
* `dense_vector` column for ANN, 
* `token_vectors` as base64 attribute, 
* `text` as attribute
> **Note** `token_vectors` is a non-filterable attribute,, so paying for a filterable index on them would be wasteful. <br>
 Set `filterable: false` in the schema for a 50% storage cost discount.

In [28]:
# Upsert documents with vectors and attributes
# idempotent - safe to re-run, existing rows will be overwritten not duplicated

ns.write(
    upsert_rows=records,
    distance_metric='cosine_distance',
    schema={
        "token_vectors": {"type": "string", "filterable": False},
        "text": {"type": "string", "filterable": False}
    }
)

print(f"Upserted {len(records)} documents to turbopuffer")

Upserted 10000 documents to turbopuffer


### Step 5: Dense Search

In [29]:
import time

def dense_search(query, top_k=5):
    query_vector = list(dense_model.embed([query]))[0].tolist()

    start_ts = time.perf_counter()
    results = ns.query(
        rank_by=["vector", "ANN", query_vector],
        limit=top_k,
        include_attributes=['text', 'token_vectors']
    )
    end_ts = time.perf_counter()
    latency = (end_ts - start_ts) * 1000

    return results.rows, latency, results.billing

In [31]:
# dense search test
# get first query
first_query_id = list(ground_truth_ids.keys())[0]
first_query = df[df['query_id'] == first_query_id]['query'].values[0]
correct_id = ground_truth_ids[first_query_id]

print(f"Query: {first_query}")
print(f"Correct document ID: {correct_id}")
print(f"Correct passage: {sampled_corpus[correct_id - 1][:200]}")

Query: )what was the immediate impact of the success of the manhattan project?
Correct document ID: 5947
Correct passage: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of


In [33]:
results, latency, billing = dense_search(first_query, top_k=10)
print(billing)
print(f"Dense latency: {latency:.1f}ms")
for row in results:
    print(f"ID: {row.id} | dist: {row['$dist']:.4f} | {row.text[:100]}")

QueryBilling(billable_logical_bytes_queried=256000000, billable_logical_bytes_returned=448916)
Dense latency: 158.4ms
ID: 5947 | dist: 0.3557 | The presence of communication amid scientific minds was equally important to the success of the Manh
ID: 8068 | dist: 0.5659 | Introduction The nuclear arms race began during the Second World War. Even before the United States 
ID: 3058 | dist: 0.5725 | One of the main reasons Hanford was selected as a site for the Manhattan Project's B Reactor was its
ID: 8565 | dist: 0.6431 | Yalta Conference In February 1945, Roosevelt had met with Churchill and Stalin at the Soviet city of
ID: 451 | dist: 0.6607 | Therefore, the failure of the League of Nations was an important cause of the Second World War. Anot
ID: 1627 | dist: 0.6742 | Sullivan was a prolific architect; over the course of his life he had a hand in the creation of more
ID: 5677 | dist: 0.6779 | Unfortunately however, we would put some of our most vital, steamed allies at a significant str

### Step 6: ColBERT: Late Interaction
Same query, ANN top-100, client-side MaxSim re-rank

In [46]:
import numpy as np

#MaxSim function
def maxsim(query_tokens, doc_tokens):
    query_tokens = np.array(query_tokens) #(Q, 128) matrix
    doc_tokens = np.array(doc_tokens) #(D, 128) matrix
    scores = np.dot(query_tokens, doc_tokens.T) #score is (Q, D) matrix
    max_similarities = scores.max(axis=1) #max similarity score for each query token against the whole document.
    maxim_score = max_similarities.sum() #final aggregated maxim socre of the query against the document

    return maxim_score 

#Full ColBERT search function
def colbert_search(query, top_k=5):
    query_embedding = list(colbert_model.embed([query]))[0]

    #stage 1 - dense retrieval of top 100 ANN documents
    candidates, dense_latency, dense_billing = dense_search(query, top_k=100)

    #stage 2 - rerank with MaxSim
    start_ts = time.perf_counter()
    scores = []
    for candidate in candidates:
        token_vectors_bytes = base64.b64decode(candidate.token_vectors)
        doc_tokens = np.frombuffer(token_vectors_bytes, dtype=np.float32).reshape(-1, 128)
        score = maxsim(query_embedding, doc_tokens)
        scores.append((score, candidate.text, candidate.id))

    scores.sort(key=lambda x: x[0], reverse=True)
    end_ts = time.perf_counter()
    colbert_latency = (end_ts - start_ts) * 1000

    return scores[:top_k], dense_latency, colbert_latency, dense_billing


In [58]:
results, dense_l, rerank_l, billing = colbert_search(first_query, top_k=10)
print(billing)
print(f"ColBERT total latency: {dense_l:.1f} + {rerank_l:.1f} = {dense_l + rerank_l:.1f}ms")
for score, text, doc_id in results:
    print(f"ID: {doc_id} | score: {score:.4f} | {text[:100]}")

QueryBilling(billable_logical_bytes_queried=256000000, billable_logical_bytes_returned=4308460)
ColBERT total latency: 398.4 + 12.4 = 410.7ms
ID: 5947 | score: 10.8438 | The presence of communication amid scientific minds was equally important to the success of the Manh
ID: 3058 | score: 7.9555 | One of the main reasons Hanford was selected as a site for the Manhattan Project's B Reactor was its
ID: 4537 | score: 7.5932 | Quick Answer. The primary effect of manifest destiny is that the United States is a bi-coastal natio
ID: 2338 | score: 7.4412 | The economic impact of the Great Depression was enormous, including both extreme human suffering and
ID: 2095 | score: 7.0809 | This paper. highlights the types of construction delays due to which project suffer time and cost ov
ID: 8068 | score: 7.0722 | Introduction The nuclear arms race began during the Second World War. Even before the United States 
ID: 8279 | score: 7.0231 | Domestic accomplishments of President Kennedy: 1  Promoted the

### Step 7: Evaluation
#### Recall@K Evaluation

To quantitatively compare dense retrieval and ColBERT late interaction, we use **Recall@K**, the fraction of queries where the known correct answer appears in the top K results.

The Quora duplicates dataset gives us ground truth: each `anchor` question has a known `positive` duplicate in the corpus. A perfect retrieval system should return the duplicate as rank #1.

We evaluate across 25 queries at two thresholds:

- **Recall@1** — did the correct answer come back as rank #1? Measures ranking precision.
- **Recall@5** — did the correct answer appear anywhere in the top 5? Measures retrieval coverage.


In [53]:
def evaluate(top_k=5, n=25):
    dense_hits = 0
    colbert_hits = 0
    
    for query_id, correct_id in list(ground_truth_ids.items())[:n]:
        query = df[df['query_id'] == query_id]['query'].values[0]
        
        # dense search
        dense_results, _, _ = dense_search(query, top_k=top_k)
        dense_ids = [row.id for row in dense_results]
        if correct_id in dense_ids:
            dense_hits += 1
        
        # colbert search
        colbert_results, _, _, _ = colbert_search(query, top_k=top_k)
        colbert_ids = [doc_id for _, _, doc_id in colbert_results]
        if correct_id in colbert_ids:
            colbert_hits += 1
    
    print(f"Dense Recall@{top_k}:   {dense_hits/n:.2f} ({dense_hits}/{n})")
    print(f"ColBERT Recall@{top_k}: {colbert_hits/n:.2f} ({colbert_hits}/{n})")

In [54]:
#Recall@5 over 25 samples
evaluate(top_k=5, n=25)

Dense Recall@5:   1.00 (25/25)
ColBERT Recall@5: 1.00 (25/25)


In [50]:
#Recall@1 over 50 samples
evaluate(top_k=1, n=50)

Dense Recall@1:   0.92 (46/50)
ColBERT Recall@1: 0.92 (46/50)


In [51]:
#Recall@1 over 100 samples
evaluate(top_k=1, n=100)

Dense Recall@1:   0.91 (91/100)
ColBERT Recall@1: 0.91 (91/100)


In [52]:
#Recall@5 over 100 samples
evaluate(top_k=5, n=100)


Dense Recall@5:   0.99 (99/100)
ColBERT Recall@5: 1.00 (100/100)


In [ ]:
#Latency evaluation
dense_search("what was the immediate impact of the manhattan project")
results, dense_latency, billing = dense_search("what was the immediate impact of the manhattan project")
print(f"Dense warm latency: {dense_latency:.1f}ms")

scored, dense_l, rerank_l, billing = colbert_search("what was the immediate impact of the manhattan project")
print(f"ColBERT total latency: {dense_l:.1f} + {rerank_l:.1f} = {dense_l + rerank_l:.1f}ms")

Dense warm latency: 43.1ms
ColBERT total latency: 255.1 + 11.8 + 266.9ms


## Analysis: Dense Retrieval vs ColBERT Late Interaction

### Setup
- **Corpus:** 1,000 questions from the Quora Duplicates dataset + 10,000 passages from MS MARCO v2.1
- **Dense model:** `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions)
- **ColBERT model:** `colbert-ir/colbertv2.0` (128 dimensions per token)
- **Evaluation:** Does the known correct answer appear in the top K results?

---

### Result Quality

> **TLDR:** Dense won on Quora. ColBERT ties dense on MS MARCO. The dataset matters more than the model.

#### Quora Duplicates (1K single-sentence questions)

| Metric | Dense (MiniLM) | ColBERT |
|---|---|---|
| Recall@1 (n=50) | 0.94 (47/50) | 0.84 (42/50) |
| Recall@5 (n=25) | 1.00 (25/25) | 1.00 (25/25) |

Dense won. Quora is paraphrases — questions that mean the same thing in slightly different words. Dense retrieval handles this well. ColBERT's token-level matching got confused by individual word matches out of context — "sleep" matching "sleeping" in an unrelated document.

Also, each document is one sentence. ColBERT was designed for passage retrieval — longer documents with multiple competing ideas. Single sentences don't give it enough to work with.

#### MS MARCO v2.1 (10K multi-sentence passages, real search queries)

| Metric | Dense (MiniLM) | ColBERT |
|---|---|---|
| Recall@1 (n=100) | 0.91 (91/100) | 0.91 (91/100) |
| Recall@5 (n=100) | 0.99 (99/100) | 1.00 (100/100) |

Tied at Recall@1. ColBERT wins by one at Recall@5 — it found the correct answer in the top 5 for every single query.

MS MARCO is the dataset ColBERT was originally benchmarked on. Multi-sentence passages, real Bing queries, multiple competing ideas per document. This is ColBERT's home turf. The gap is small here at 10K documents — it would widen at production scale where ANN recall starts dropping.

**The key finding:** dataset choice matters more than model choice. ColBERT doesn't universally beat dense — it wins on the right data.

---

### Latency

> **TLDR:** 2-stage ColBERT with client-side MaxSim calculation is ~6x slower on MS MARCO (267ms vs 44ms). And MaxSim dropped to just 12ms. base64 float32 decoding is faster than JSON deserialization

| Approach | Fetch | MaxSim | Total | Dataset |
|---|---|---|---|---|
| Dense only | 70ms | 0ms | 70ms | Quora |
| ColBERT float32 JSON | 275ms | 50ms | 325ms | Quora |
| ColBERT int8 JSON | 188ms | 27ms | 214ms | Quora |
| Dense only | 44ms | 0ms | 44ms | MS MARCO |
| ColBERT base64 float32 | 255ms | 12ms | 267ms | MS MARCO |

## D2: Roadmap: How turbopuffer Could Better Support Late Interaction

Building this guide exposed two places where turbopuffer could make late 
interaction significantly easier and faster to implement.

---

### 1. Native Multi-Vector Column Support

Right now I had to store token vectors as a JSON string attribute, it works 
but it's slow to fetch and deserialize at query time. This added ~470ms to 
every ColBERT query in my testing.

If turbopuffer supported a native multi-vector column type, token matrices 
could be stored in compact binary format instead of JSON strings. This would 
be the single biggest latency improvement for late interaction on turbopuffer today.

```python
# what this could look like
schema={
    "token_vectors": {
        "type": "multi_vector",
        "dimensions": 128,
        "filterable": False
    }
}
```

---

### 2. Server-Side MaxSim Scoring

Right now MaxSim reranking happens in my Python code after fetching results 
from turbopuffer. It works but it means pulling 100 documents worth of token 
vectors over the network on every query.

Server-side MaxSim would be the natural equivalent for multi-vector columns.
Colocate compute and data, return only the final ranked results.

```python
# what this could look like
results = ns.query(
    rank_by=["token_vectors", "MaxSim", query_token_vectors],
    limit=10
)
```

---

### Why This Matters

Weaviate already supports native multi-vector and server-side MaxSim. 
turbopuffer's object storage architecture makes late interaction dramatically cheaper, 
native multi-vector support would make it faster too.